# 🎧 SPEEDWAVE: Audio Speed Converter (Google Colab)

Runs SpeedWave on a free Google Colab machine. **No installation on your own computer.**

**How to use (Cara guna):**
1. Click **Runtime → Run all** (or `Ctrl+F9`). *Klik Runtime → Run all.*
2. Wait 2–4 minutes for setup. *Tunggu 2–4 minit.*
3. Click the **🚀 Open SpeedWave** button that appears in step 3. *Klik butang Open SpeedWave.*

The Google link only works in **this browser**, signed in to the same Google account.
To open it on your phone or another browser, run step 4 (Cloudflare public link).

> The server keeps running while this Colab tab is open. Colab shuts down after a period of inactivity
> (about 90 minutes) or about 12 hours at most. Files are deleted when the runtime stops.

In [ ]:
#@title 1️⃣ Install SpeedWave (FFmpeg + Rubber Band, Python, Node)  { display-mode: "form" }
BRANCH = "claude/audio-speed-converter-ds1og6"  #@param {type:"string"}
REPO = "https://github.com/yuudayo12/AM-Marketplace.git"

import os, re, shutil, subprocess, sys

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if check and r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise SystemExit(f"❌ Failed: {cmd}")
    return r

os.chdir("/content")
print("⬇️  Downloading SpeedWave…")
shutil.rmtree("/content/AM-Marketplace", ignore_errors=True)
sh(f"git clone -q --depth 1 -b {BRANCH} {REPO} /content/AM-Marketplace")
APP = "/content/AM-Marketplace/speedwave"

# FFmpeg: the static build includes librubberband 3.x (high-quality R3 engine).
# If it can't be downloaded, fall back to Ubuntu's ffmpeg (Rubber Band R2, still pitch-preserving).
def ffmpeg_ok(path):
    r = sh(f"{path} -hide_banner -filters", check=False)
    e = sh(f"{path} -hide_banner -encoders", check=False)
    return r.returncode == 0 and " rubberband " in r.stdout and "libmp3lame" in e.stdout and "libvorbis" in e.stdout

FF, FP = "/content/ffmpeg/ffmpeg", "/content/ffmpeg/ffprobe"
if not ffmpeg_ok(FF):
    print("⬇️  Installing FFmpeg (static build with Rubber Band)…")
    os.makedirs("/content/ffmpeg", exist_ok=True)
    sh("curl -fsSL https://johnvansickle.com/ffmpeg/releases/ffmpeg-release-amd64-static.tar.xz"
       " | tar xJ -C /content/ffmpeg --strip-components=1", check=False)
if not ffmpeg_ok(FF):
    print("⚠️  Static FFmpeg unavailable; installing Ubuntu ffmpeg instead…")
    sh("apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null")
    FF, FP = shutil.which("ffmpeg"), shutil.which("ffprobe")
    if not ffmpeg_ok(FF):
        raise SystemExit("❌ No FFmpeg with rubberband/libmp3lame/libvorbis found.")
print("✅", sh(f"{FF} -version").stdout.splitlines()[0])

print("⬇️  Installing Python packages…")
sh(f"{sys.executable} -m pip install -q -r {APP}/backend/requirements.txt")

# Node.js 18+ is needed to build the web interface.
NODE_BIN = ""
r = sh("node --version", check=False)
major = int(re.sub(r"[^0-9.]", "", r.stdout).split(".")[0] or 0) if r.returncode == 0 else 0
if major < 18:
    print("⬇️  Installing Node.js 22…")
    sh("mkdir -p /content/node && curl -fsSL https://nodejs.org/dist/v22.11.0/node-v22.11.0-linux-x64.tar.xz"
       " | tar xJ -C /content/node --strip-components=1")
    NODE_BIN = "/content/node/bin"
    os.environ["PATH"] = NODE_BIN + ":" + os.environ["PATH"]
print("🛠️  Building the web interface…")
sh(f"cd {APP}/frontend && npm install --no-audit --no-fund --silent && npm run build --silent")
print("\n✅ Setup complete. Run step 2.")

In [ ]:
#@title 2️⃣ Start the SpeedWave server  { display-mode: "form" }
MAX_UPLOAD_MB = 1000  #@param {type:"integer"}
import json, os, subprocess, time, urllib.request

subprocess.run("pkill -f 'uvicorn app.main:app'", shell=True)
env = dict(os.environ,
           SPEEDWAVE_FFMPEG=FF, SPEEDWAVE_FFPROBE=FP,
           SPEEDWAVE_DATA_DIR="/content/speedwave-data",
           SPEEDWAVE_MAX_UPLOAD_MB=str(MAX_UPLOAD_MB),
           SPEEDWAVE_FILE_TTL_MINUTES="240")
log = open("/content/speedwave.log", "w")
server = subprocess.Popen([sys.executable, "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
                          cwd=f"{APP}/backend", env=env, stdout=log, stderr=subprocess.STDOUT)
health = None
for _ in range(60):
    try:
        health = json.load(urllib.request.urlopen("http://127.0.0.1:8000/api/health", timeout=2))
        break
    except Exception:
        if server.poll() is not None:
            break
        time.sleep(0.5)
if not health:
    print(open("/content/speedwave.log").read()[-3000:])
    raise SystemExit("❌ The server did not start (log above).")
print("✅ Server running")
print("   FFmpeg:     ", health["ffmpeg"])
print("   Rubber Band:", health["rubberband_engine"])

In [ ]:
#@title 3️⃣ 🚀 Open SpeedWave (Google link)  { display-mode: "form" }
from google.colab.output import eval_js
from IPython.display import HTML, display

URL = eval_js("google.colab.kernel.proxyPort(8000, {'cache': false})")
display(HTML(f"""
<div style="font-family:Inter,system-ui,sans-serif;padding:18px;border-radius:14px;background:#15171d;border:1px solid #2e323d;max-width:640px">
  <div style="font-weight:800;letter-spacing:.2em;color:#c4b5fd;font-size:18px">SPEEDWAVE</div>
  <div style="color:#94a3b8;margin:6px 0 14px">Your app is ready. It opens in a new tab.</div>
  <a href="{URL}" target="_blank" style="display:inline-block;padding:12px 22px;border-radius:12px;color:#fff;font-weight:700;
     text-decoration:none;background:linear-gradient(90deg,#7c3aed,#3b82f6)">🚀 Open SpeedWave</a>
  <div style="color:#64748b;font-size:12px;margin-top:12px;word-break:break-all">{URL}</div>
</div>"""))

### Optional: public link (phone / other browser)
The Google link above works only in this browser. Step 4 creates a temporary public
`https://….trycloudflare.com` link instead. **Anyone with that link can use your SpeedWave while it runs**
(each visitor still only sees their own files), so share it only with people you trust.

In [ ]:
#@title 4️⃣ (Optional) Create a public link with Cloudflare  { display-mode: "form" }
import re, subprocess, time
if not os.path.exists("/content/cloudflared"):
    sh("curl -fsSL -o /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
       " && chmod +x /content/cloudflared")
subprocess.run("pkill -f 'cloudflared tunnel'", shell=True)
tunnel_log = open("/content/cloudflared.log", "w")
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
                          stdout=tunnel_log, stderr=subprocess.STDOUT)
public = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/cloudflared.log").read())
    if m:
        public = m.group(0); break
    time.sleep(0.5)
if public:
    display(HTML(f'<a href="{public}" target="_blank" style="font-size:16px;font-weight:700">🌍 {public}</a>'))
else:
    print("Could not create a tunnel. Log:\n", open("/content/cloudflared.log").read()[-2000:])

In [ ]:
#@title 🔧 Troubleshooting: show server log / stop server  { display-mode: "form" }
ACTION = "show log"  #@param ["show log", "stop server"]
if ACTION == "show log":
    print(open("/content/speedwave.log").read()[-5000:])
else:
    subprocess.run("pkill -f 'uvicorn app.main:app'; pkill -f 'cloudflared tunnel'", shell=True)
    print("Stopped.")